# EggSizeGrader — Day 38 / 100 Days of Projects

**Task:** 3-class egg-size **object detection** (large / medium / small), with YOLO bounding boxes. This is an *image-based category detector*, **not** a calibrated measurement of grams, physical diameter, or regulatory egg grades.

**Data mount:** `/kaggle/input/datasets/loliwibu/eggsizegrader-dataset`  
**Protocol:** discovery → source/annotation audit → source-identity leakage check → deterministic group-disjoint resplit → baseline → train & validation → validation-only champion → once-only held-out test → real predictions → export, model card, manifest and ZIP.

**Integrity notes:** The original uploaded data are expected to contain `train/`, `valid/`, `test/`, `data.yaml`. Earlier audit identified two distinct upstream ZIP projects; only a 3-class `large, medium, small` YOLO source may be used here. If this Kaggle input differs, the notebook **stops**, rather than guessing labels. No metrics below are pre-filled. The notebook may require Kaggle Internet for installing Ultralytics/downloading pretrained weights.

In [ ]:
# 0 — Environment, seed & reproducibility
import os, sys, json, math, random, re, shutil, subprocess, importlib.util, hashlib, platform, time, csv, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
INPUT = Path('/kaggle/input/datasets/loliwibu/eggsizegrader-dataset')
OUT = Path('/kaggle/working/EggSizeGrader_Day38')
REPORTS, PLOTS, EXAMPLES, MODELS = (OUT / s for s in ('reports','assets/plots','assets/examples','models'))
for p in (REPORTS, PLOTS, EXAMPLES, MODELS): p.mkdir(parents=True, exist_ok=True)
assert INPUT.is_dir(), f'Missing Kaggle input: {INPUT}. Attach EggSizeGrader dataset in Add Input.'
print({'python':sys.version.split()[0], 'platform':platform.platform(), 'input':str(INPUT), 'timestamp_utc':datetime.now(timezone.utc).isoformat()})
print('Input files (first 30):')
for p in sorted(INPUT.rglob('*'))[:30]: print(p.relative_to(INPUT))

## 1 — Dataset discovery & provenance

Identify the actual single root containing `data.yaml` and split folders. Confirm class names in YAML, record original split membership, and never copy unexplained source files into the training set. The upstream Roboflow `EGGS SIZES FINAL v2` was reported as CC BY 4.0 in source YAML; this code records what the **mounted YAML actually says**. Attribution and any restrictions must be checked against the original source.

In [ ]:
# 1 — Find YAML and verify the intended task / label order
if importlib.util.find_spec('yaml') is None:
    subprocess.check_call([sys.executable, '-m','pip','install','-q','pyyaml'])
import yaml
candidates = []
for y in INPUT.rglob('data.yaml'):
    if all((y.parent / x / 'images').is_dir() for x in ('train','test')) and ((y.parent / 'valid' / 'images').is_dir() or (y.parent/'val'/'images').is_dir()):
        candidates.append(y)
assert len(candidates) == 1, f'Expected exactly 1 YOLO data.yaml root; got {len(candidates)}: {candidates}'
yaml_file = candidates[0]; ROOT = yaml_file.parent
cfg = yaml.safe_load(yaml_file.read_text(encoding='utf-8-sig'))
names = cfg.get('names')
if isinstance(names, dict): names = [names[k] for k in sorted(names, key=lambda x:int(x))]
assert isinstance(names,list), f'Invalid class names: {names}'
names = [str(s).strip().lower() for s in names]
assert names == ['large','medium','small'] and cfg.get('nc',3) == 3, f'Wrong task/classes: {names}. Need 3-class egg-size labels.'
SPLITS = {'train':'train','val':'valid' if (ROOT/'valid'/'images').is_dir() else 'val','test':'test'}
provenance = {'input_mount':str(INPUT),'dataset_root':str(ROOT),'source_yaml':str(yaml_file.relative_to(INPUT)), 'classes':names,'nc':3,'roboflow':cfg.get('roboflow',{}),'source_splits':SPLITS,'accessed_utc':datetime.now(timezone.utc).isoformat(), 'license_verification':'source YAML declaration only; independently verify upstream rights'}
(REPORTS/'dataset_provenance.json').write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(provenance,indent=2,ensure_ascii=False))

## 2 — Pair images & labels; audit YOLO coordinates

YOLO detection labels use `class_id cx cy width height`, normalized to [0,1]. Empty label files are allowed (background examples); missing label files, invalid coordinates, unknown IDs, and unreadable images are **not silently repaired**.

In [ ]:
# 2 — Validate every image and label
EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
records=[]; issues=[]; class_objects=Counter(); source_split_counts={}
for split, dirname in SPLITS.items():
    img_dir=ROOT/dirname/'images'; lab_dir=ROOT/dirname/'labels'
    assert img_dir.is_dir() and lab_dir.is_dir(),f'Missing images/labels for {split}'
    imgs=sorted(p for p in img_dir.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
    source_split_counts[split]=len(imgs)
    for im in imgs:
        lab=lab_dir/(im.relative_to(img_dir).with_suffix('.txt'))
        if not lab.is_file(): issues.append({'image':str(im),'issue':'missing_label'}); continue
        try:
            with Image.open(im) as obj:
                obj.verify()
            with Image.open(im) as obj:
                width,height=obj.size
            if width<=0 or height<=0: raise ValueError('zero dimensions')
        except Exception as ex: issues.append({'image':str(im),'issue':f'bad_image:{ex}'}); continue
        counts=Counter()
        for lineno,line in enumerate(lab.read_text(encoding='utf-8-sig').splitlines(),1):
            if not line.strip(): continue
            fields=line.strip().split()
            try:
                if len(fields)!=5: raise ValueError(f'expected 5 fields, got {len(fields)}')
                class_float=float(fields[0]); cid=int(class_float)
                if class_float!=cid or cid not in range(3): raise ValueError('invalid class id')
                cx,cy,w,h=map(float,fields[1:])
                if not all(math.isfinite(v) for v in [cx,cy,w,h]): raise ValueError('non-finite box')
                eps=1e-4
                if not (0<=cx<=1 and 0<=cy<=1 and 0<w<=1 and 0<h<=1): raise ValueError('out of range')
                if cx-w/2 < -eps or cy-h/2 < -eps or cx+w/2 > 1+eps or cy+h/2 > 1+eps: raise ValueError('box beyond boundary')
                counts[cid]+=1; class_objects[(split,names[cid])]+=1
            except Exception as ex: issues.append({'image':str(im),'label':str(lab),'line':lineno,'issue':str(ex)})
        records.append({'split_original':split,'image':str(im),'label':str(lab),'stem':im.stem, 'width':width,'height':height, 'objects':sum(counts.values()), **{f'objects_{name}':counts.get(k,0) for k,name in enumerate(names)}})
audit_df=pd.DataFrame(records)
pd.DataFrame(issues,columns=['image','label','line','issue']).to_csv(REPORTS/'annotation_issues.csv',index=False)
assert len(audit_df)>0,'No usable images found'
assert not issues, f'{len(issues)} label/image errors found. Inspect reports/annotation_issues.csv; STOP rather than train on invalid labels. Examples: {issues[:3]}'
assert sum(audit_df['objects'])>0, 'No annotated objects.'
audit_df.to_csv(REPORTS/'source_inventory.csv',index=False)
print('Original split sizes:',source_split_counts,'valid image/label pairs:',len(audit_df))
print('Annotated objects by original split/class:')
print(pd.DataFrame([{'split':k[0],'class':k[1],'objects':v} for k,v in sorted(class_objects.items())]).to_string(index=False))
print('Empty/background images:',int((audit_df.objects==0).sum()))

## 3 — Leakage audit with source identities and exact hashes

Roboflow filenames commonly follow `original_name_jpg.rf.<augmentation/hash>.jpg`; augmented siblings can otherwise contaminate train/test. We conservatively strip the `.rf.*` suffix, compare SHA-256 bytes, and **rebuild deterministic group-disjoint splits from all mounted images**. Identical bytes and identical source stems are unioned; labels of exact duplicates must match or we stop. Near-duplicate *visual* leakage cannot be completely ruled out by filename and byte hashing; manual checks remain necessary.

**Important:** We do not use the source test split to choose a model. The new held-out test membership is frozen before training. Source split membership is preserved in the audit.

In [ ]:
# 3 — Content hashes / Roboflow source keys / union-find groups
source_regex=re.compile(r'(?i)(?:_jpe?g|_png|_bmp|_webp)?\.rf\.[0-9a-z]+$')
def source_key(stem): return source_regex.sub('',stem).casefold().strip()
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def label_canonical(path):
    return tuple(sorted(' '.join(line.split()) for line in Path(path).read_text(encoding='utf-8-sig').splitlines() if line.strip()))

assert audit_df['image'].is_unique
# Avoid costly hashing duplicate sources twice; image files here are small (~318 images).
audit_df['source_key']=audit_df['stem'].map(source_key)
audit_df['sha256']=[sha256_file(p) for p in audit_df.image]
audit_df['label_signature']=[label_canonical(p) for p in audit_df.label]
# Exact same bytes with disagreeing annotations are a source conflict.
conflicting=[]
for h,g in audit_df.groupby('sha256'):
    if len(g)>1 and g.label_signature.nunique()>1:
        conflicting.append({'hash':h,'images':g.image.tolist(),'labels_differ':True})
(REPORTS/'duplicate_label_conflicts.json').write_text(json.dumps(conflicting,indent=2),encoding='utf-8')
assert not conflicting, f'{len(conflicting)} exact image duplicates with conflicting labels! Resolve upstream before training.'

parent=list(range(len(audit_df)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[b]=a
for col in ('source_key','sha256'):
    lead={}
    for i,key in enumerate(audit_df[col]):
        if key in lead: union(i,lead[key])
        else: lead[key]=i
audit_df['group_id']=[find(i) for i in range(len(audit_df))]
source_cross=(audit_df.groupby('source_key').split_original.nunique()>1).sum()
exact_cross=(audit_df.groupby('sha256').split_original.nunique()>1).sum()
components=audit_df.group_id.nunique()
leakage_audit={'original_split_overlap_source_keys':int(source_cross),'original_split_overlap_exact_sha256':int(exact_cross),'total_images':len(audit_df),'total_conservative_groups':int(components),'method':'union(source stem stripped of Roboflow suffix, SHA256 bytes); resplit by group; visual near-duplicate not excluded'}
(REPORTS/'leakage_audit.json').write_text(json.dumps(leakage_audit,indent=2),encoding='utf-8')
print(json.dumps(leakage_audit,indent=2))
assert components>=10,f'Only {components} independent source groups: insufficient for a meaningful 3-way evaluation.'

## 4 — Frozen group-disjoint train/validation/test

Use `GroupShuffleSplit` with seed 42 (train ~70%, validation ~15%, test ~15%). Whole source groups stay together; actual proportions depend on group size. **Each split must contain all three classes** to permit a useful per-class evaluation, otherwise the notebook stops for human review. This is a group-disjoint benchmark on this dataset, **not proof of generalization to another farm, camera, flock, or egg grading standard**.

In [ ]:
# 4 — Create fresh leakage-controlled split in Kaggle working dir (do not modify mounted input)
from sklearn.model_selection import GroupShuffleSplit
idx=np.arange(len(audit_df))
gss1=GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=SEED)
tr_idx,hold_idx=next(gss1.split(idx, groups=audit_df.group_id))
gss2=GroupShuffleSplit(n_splits=1,test_size=0.50,random_state=SEED+1)
val_rel,test_rel=next(gss2.split(hold_idx,groups=audit_df.iloc[hold_idx].group_id))
val_idx,test_idx=hold_idx[val_rel],hold_idx[test_rel]
audit_df['split_clean']=''
for split, ids in [('train',tr_idx),('val',val_idx),('test',test_idx)]:
    audit_df.loc[ids,'split_clean']=split
assert (audit_df.groupby('group_id').split_clean.nunique()==1).all(), 'Group leakage after split!'
assert (audit_df.groupby('sha256').split_clean.nunique()==1).all(), 'Exact duplicate leakage after split!'
assert (audit_df.groupby('source_key').split_clean.nunique()==1).all(), 'Source name leakage after split!'
class_distribution=pd.DataFrame([{'split':split, 'images':int((audit_df.split_clean==split).sum()), **{name:int(audit_df.loc[audit_df.split_clean==split,f'objects_{name}'].sum()) for name in names}} for split in ('train','val','test')])
print(class_distribution.to_string(index=False))
class_distribution.to_csv(REPORTS/'clean_split_distribution.csv',index=False)
audit_df.drop(columns='label_signature').to_csv(REPORTS/'split_membership.csv',index=False)
assert all((class_distribution[name]>0).all() for name in names), 'At least one split lacks a class. Inspect group sizes/class coverage; no training until defensible split reviewed.'
DATA=OUT/'prepared_dataset'
for split in ('train','val','test'):
    (DATA/split/'images').mkdir(parents=True,exist_ok=True)
    (DATA/split/'labels').mkdir(parents=True,exist_ok=True)
    for r in audit_df[audit_df.split_clean==split].itertuples():
        # Stable unique filename even if originals from different source splits share a stem
        filename=hashlib.sha256(r.image.encode()).hexdigest()[:12]+'_'+Path(r.image).name
        shutil.copy2(r.image,DATA/split/'images'/filename)
        shutil.copy2(r.label,DATA/split/'labels'/(Path(filename).stem+'.txt'))
(DATA/'data.yaml').write_text(yaml.safe_dump({'path':str(DATA),'train':'train/images','val':'val/images','test':'test/images','nc':len(names),'names':names},sort_keys=False),encoding='utf-8')
print('Prepared:', DATA/'data.yaml')

## 5 — EDA: class counts and annotated image preview

Only visualize **real** annotations. Pixel-level segmentation metrics do not apply: this dataset uses object-level bounding-box labels.

In [ ]:
# 5 — EDA outputs
fig,axes=plt.subplots(1,2,figsize=(12,4))
class_distribution.set_index('split')[names].plot(kind='bar',ax=axes[0],rot=0,title='Annotated objects by split')
axes[0].set_ylabel('Objects')
audit_df.objects.plot.hist(bins=min(20,max(3,len(audit_df)//5)),ax=axes[1],title='Boxes per image')
axes[1].set_xlabel('Annotated objects')
fig.tight_layout();fig.savefig(PLOTS/'dataset_eda.png',dpi=160);plt.show()

def draw_gt(image_path,label_path):
    with Image.open(image_path) as im: im=ImageOps.exif_transpose(im).convert('RGB')
    d=ImageDraw.Draw(im); w,h=im.size
    for line in Path(label_path).read_text().splitlines():
        if not line.strip(): continue
        c,cx,cy,bw,bh=map(float,line.split()); c=int(c)
        xy=[(cx-bw/2)*w,(cy-bh/2)*h,(cx+bw/2)*w,(cy+bh/2)*h]
        d.rectangle(xy,outline=['red','blue','green'][c],width=max(2,w//250))
        d.text((xy[0],max(0,xy[1]-14)),names[c],fill=['red','blue','green'][c])
    return im
examples=sorted((DATA/'train'/'images').glob('*'))[:6]
fig,axs=plt.subplots(2,3,figsize=(12,8))
for ax in axs.flat: ax.axis('off')
for ax,p in zip(axs.flat,examples):
    lab=DATA/'train'/'labels'/(p.stem+'.txt')
    ax.imshow(draw_gt(p,lab)); ax.set_title('Real GT: '+p.name[:20],fontsize=9)
fig.tight_layout(); fig.savefig(EXAMPLES/'ground_truth_samples.png',dpi=130);plt.show()

## 6 — Baseline and training configuration

**Baselines:** a nonlearned zero-detection reference is recorded as a *sanity check*, not a YOLO mAP estimate. YOLOv8n is the single pretrained detector candidate; validation mAP50–95 selects the epoch via `best.pt`. This means the **held-out test is used only once**, after the validation-selected checkpoint is fixed. We do not manufacture additional model comparisons. For a stronger future experiment, compare against larger variants based on validation only.

Kaggle accelerator: GPU is recommended. If `ultralytics` is unavailable and Internet is disabled, enable Internet or attach an offline wheel and weights; the notebook fails clearly.

In [ ]:
# 6 — Install/check dependencies
if importlib.util.find_spec('ultralytics') is None:
    print('Ultralytics not pre-installed; attempting pip installation (Kaggle Internet may be required).')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics>=8.3,<9'])
import torch, ultralytics
from ultralytics import YOLO
print('torch:',torch.__version__,'cuda:',torch.cuda.is_available(),'ultralytics:',ultralytics.__version__)
DEVICE=0 if torch.cuda.is_available() else 'cpu'
EPOCHS=60; PATIENCE=15; IMGSZ=640; BATCH=16 if DEVICE!= 'cpu' else 4
settings={'seed':SEED,'device':DEVICE,'epochs':EPOCHS,'patience':PATIENCE,'imgsz':IMGSZ,'batch':BATCH,'optimizer':'auto','model':'yolov8n.pt','task':'detect','champion':'best epoch on validation fitness (Ultralytics) / mAP50-95'}
(REPORTS/'experiment_config.json').write_text(json.dumps(settings,indent=2),encoding='utf-8')
print(json.dumps(settings,indent=2))

In [ ]:
# 7 — Honest nonlearned baseline: predict zero boxes, recall 0 for annotated objects
baseline={'name':'predict_no_objects','number_predictions':0,'gt_objects_train':int(class_distribution.loc[class_distribution.split=='train',names].sum(axis=1).iloc[0]), 'gt_objects_val':int(class_distribution.loc[class_distribution.split=='val',names].sum(axis=1).iloc[0]), 'recall_on_annotated_objects':0.0, 'note':'Sanity baseline only; not comparable to detection mAP and no fabricated mAP is claimed.'}
(REPORTS/'baseline.json').write_text(json.dumps(baseline,indent=2),encoding='utf-8')
print(baseline)

## 7 — Train on `train`, select using `val`

`best.pt` is selected internally using validation fitness. If weights download is blocked, provide `yolov8n.pt` in a mounted Kaggle input dataset (or enable Kaggle Internet). Do not substitute a random model and report it as pretrained. The training log is real output from this run.

In [ ]:
# 8 — Fit model (never touch held-out test for tuning)
weights_candidates=list(INPUT.rglob('yolov8n.pt'))
WEIGHTS=str(weights_candidates[0]) if weights_candidates else 'yolov8n.pt'
print('Weights source:',WEIGHTS)
model=YOLO(WEIGHTS)
TRAIN_DIR=OUT/'training'
train_result=model.train(data=str(DATA/'data.yaml'),epochs=EPOCHS,patience=PATIENCE,imgsz=IMGSZ,batch=BATCH,device=DEVICE,seed=SEED,deterministic=True,workers=2,project=str(TRAIN_DIR),name='yolov8n',exist_ok=True,plots=True,verbose=True)
BEST=TRAIN_DIR/'yolov8n'/'weights'/'best.pt'
assert BEST.is_file(),f'Missing validation-selected best.pt: {BEST}'
shutil.copy2(BEST,MODELS/'best.pt')
print('Saved trained checkpoint:', MODELS/'best.pt', 'bytes:',BEST.stat().st_size)

## 8 — Validation evaluation and locked champion

Explicitly evaluate the selected checkpoint on the validation split. Save per-class AP, metric values, and validation confusion matrix when available. Model selection criterion is validation only. No held-out test value is consulted here.

In [ ]:
# 9 — Validation metrics and selected model provenance
best_model=YOLO(str(BEST))
val_metrics=best_model.val(data=str(DATA/'data.yaml'),split='val',imgsz=IMGSZ,batch=BATCH,device=DEVICE,project=str(OUT/'eval'),name='validation',exist_ok=True,plots=True,verbose=False)
def metrics_dict(res):
    out={'precision':float(res.box.mp),'recall':float(res.box.mr),'mAP50':float(res.box.map50),'mAP50_95':float(res.box.map)}
    for i,name in enumerate(names):
        v=float(res.box.maps[i]) if i<len(res.box.maps) else None
        out[f'AP50_95_{name}']=v
    return out
validation=metrics_dict(val_metrics)
selection={'selected_checkpoint':'models/best.pt','selected_by':'Ultralytics training validation fitness; validation mAP50-95 also reported','validation':validation,'test_used_for_selection':False,'num_candidate_architectures':1}
(REPORTS/'validation_metrics.json').write_text(json.dumps(validation,indent=2),encoding='utf-8')
(REPORTS/'model_selection.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
print(json.dumps(selection,indent=2))

## 9 — Final held-out test (once)

Evaluate `best.pt` **once on `test`**, after champion selection. Do not tune confidence, image size, training epochs, class mappings, or augmentation using these results. Metrics are object-detection precision/recall and mAP; **not image classification accuracy**.

In [ ]:
# 10 — One-time final test
assert (REPORTS/'model_selection.json').is_file()
test_metrics=best_model.val(data=str(DATA/'data.yaml'),split='test',imgsz=IMGSZ,batch=BATCH,device=DEVICE,project=str(OUT/'eval'),name='heldout_test',exist_ok=True,plots=True,verbose=False)
test_result=metrics_dict(test_metrics)
heldout={'protocol':'one-time group-disjoint held-out test','test_images':int((audit_df.split_clean=='test').sum()),'test_groups':int(audit_df.loc[audit_df.split_clean=='test','group_id'].nunique()),'checkpoint':'models/best.pt', 'metrics':test_result,'claim_boundary':'dataset-specific object-detection performance; physical size/weight in real units not evaluated'}
(REPORTS/'heldout_test_metrics.json').write_text(json.dumps(heldout,indent=2),encoding='utf-8')
pd.DataFrame([{'class':name,'AP50_95':test_result[f'AP50_95_{name}']} for name in names]).to_csv(REPORTS/'heldout_per_class_ap.csv',index=False)
print(json.dumps(heldout,indent=2))

## 10 — Real predictions and qualitative error review

Save actual annotated inference images and a small prediction summary from the frozen test set. These are genuine model predictions. Each output is labeled as a prediction, **not** ground truth or physical size measurement. Inspect false positives and missed/overlapping objects manually.

In [ ]:
# 11 — Inference examples on held-out images
TEST_IMAGES=sorted((DATA/'test'/'images').glob('*'))
assert TEST_IMAGES
sample_images=TEST_IMAGES[:min(12,len(TEST_IMAGES))]
results=best_model.predict(source=[str(p) for p in sample_images],imgsz=IMGSZ,device=DEVICE,conf=0.25,iou=0.7,verbose=False)
pred_rows=[]
for path,res in zip(sample_images,results):
    annotated=Image.fromarray(res.plot()[:,:,::-1]) # Ultralytics plot returns BGR
    annotated.save(EXAMPLES/f'pred_{path.stem}.png')
    boxes=res.boxes
    pred_rows.append({'image':path.name,'num_predictions':len(boxes),'predicted_classes':';'.join(names[int(x)] for x in boxes.cls.cpu().tolist()),'confidences':';'.join(f'{float(x):.4f}' for x in boxes.conf.cpu().tolist())})
pd.DataFrame(pred_rows).to_csv(REPORTS/'prediction_examples.csv',index=False)
fig,axes=plt.subplots(2,3,figsize=(12,8))
for ax in axes.flat: ax.axis('off')
for ax,p in zip(axes.flat,sorted(EXAMPLES.glob('pred_*.png'))[:6]):
    ax.imshow(Image.open(p));ax.set_title(p.name[:28],fontsize=8)
fig.tight_layout();fig.savefig(PLOTS/'heldout_prediction_grid.png',dpi=140);plt.show()

## 11 — Export verification (PyTorch; optional ONNX)

The verified deliverable is the trained `.pt` checkpoint. ONNX export is attempted only if requested; failures are recorded, and it is **never** claimed valid merely because a file exists. Default `EXPORT_ONNX=False` avoids environment dependencies. If enabled, a successful export must be loadable and execute one real forward pass.

In [ ]:
# 12 — Checkpoint verification and optional ONNX smoke test
EXPORT_ONNX=False
pt=MODELS/'best.pt'
assert pt.is_file() and pt.stat().st_size>100_000
reloaded=YOLO(str(pt))
check=reloaded.predict(source=str(TEST_IMAGES[0]),imgsz=IMGSZ,device=DEVICE,verbose=False)
assert len(check)==1 and check[0].boxes is not None
export_status={'pytorch_checkpoint_exists':True,'pytorch_reload_and_inference':True,'onnx_requested':EXPORT_ONNX,'onnx_verified':False}
if EXPORT_ONNX:
    try:
        exported=Path(reloaded.export(format='onnx',imgsz=IMGSZ,dynamic=False,simplify=False))
        import onnxruntime as ort
        sess=ort.InferenceSession(str(exported),providers=['CPUExecutionProvider'])
        im=Image.open(TEST_IMAGES[0]).convert('RGB').resize((IMGSZ,IMGSZ))
        inp=(np.asarray(im,dtype=np.float32).transpose(2,0,1)[None]/255.0).astype(np.float32)
        out_onnx=sess.run(None,{sess.get_inputs()[0].name:inp})
        assert len(out_onnx)>0 and all(np.all(np.isfinite(x)) for x in out_onnx)
        shutil.copy2(exported,MODELS/'best.onnx'); export_status['onnx_verified']=True
    except Exception as ex:
        export_status['onnx_error']=f'{type(ex).__name__}: {ex}'
(REPORTS/'export_verification.json').write_text(json.dumps(export_status,indent=2),encoding='utf-8')
print(export_status)

## 12 — Manifest, scientific limitations, and deliverable ZIP

Export includes the trained checkpoint, validation & held-out metrics, split membership, audit files, genuine plots/predictions and a model card. Training dataset images are **not redistributed** inside this model-artifact ZIP. The prepared dataset remains available under `/kaggle/working/EggSizeGrader_Day38/prepared_dataset/`; published data should respect the upstream license and attribution. This is a research prototype, not a certified egg-grading instrument.

In [ ]:
# 13 — Model card + manifest and artifact ZIP
card=f"""# EggSizeGrader model card — Day 38

## Intended use
Detect egg instances and assign dataset-provided visual size labels: {', '.join(names)}.
**Not** a calibrated egg mass, dimension, or formal grading instrument.

## Data
Mounted input: {INPUT}
Dataset root: {ROOT}
Source license declaration: {cfg.get('roboflow',{}).get('license','not supplied')} (verify independently)
Images: {len(audit_df)}, unique conservative source groups: {components}

## Leakage controls
Roboflow source-key and exact SHA-256 group-disjoint resplit. Original split overlaps: source keys {source_cross}; SHA256 {exact_cross}.
Visual perceptual near-duplicates and annotation correctness still require manual review.

## Training / selection
YOLOv8n detector, seed {SEED}, img size {IMGSZ}, max epochs {EPOCHS}. Validation-only best checkpoint.

## Held-out test
Test images: {heldout['test_images']}, independent group keys: {heldout['test_groups']}
Precision: {test_result['precision']:.4f}; Recall: {test_result['recall']:.4f}
mAP50: {test_result['mAP50']:.4f}; mAP50-95: {test_result['mAP50_95']:.4f}
Individual class AP values: reports/heldout_per_class_ap.csv

## Limitations
Small image dataset; approximate source-identity grouping; only dataset-specific evaluation.
No validation against independent farms, lighting, unseen cameras, mass/diameter ground truth,
or industry grade standards. Performance should not be interpreted as reliability of physical sizing.

## Reproduction
Notebook input mount listed above; check reports/experiment_config.json, leakage_audit.json,
split_membership.csv, model_selection.json, heldout_test_metrics.json, and export_verification.json.
"""
(OUT/'MODEL_CARD.md').write_text(card,encoding='utf-8')
manifest=[]
# Exclude prepared dataset and full training/evaluation copies from the deliverable; keep relevant results
for folder in (REPORTS,PLOTS,EXAMPLES,MODELS):
    for p in sorted(folder.rglob('*')):
        if p.is_file():
            manifest.append({'path':p.relative_to(OUT).as_posix(),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
p=OUT/'MODEL_CARD.md'
manifest.append({'path':p.relative_to(OUT).as_posix(),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
(REPORTS/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
zip_path=Path('/kaggle/working/EggSizeGrader_Day38_Artifacts.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for p in sorted([*REPORTS.rglob('*'),*PLOTS.rglob('*'),*EXAMPLES.rglob('*'),*MODELS.rglob('*'),OUT/'MODEL_CARD.md']):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT).as_posix())
with zipfile.ZipFile(zip_path) as z:
    bad=z.testzip(); assert bad is None,f'Corrupt ZIP entry: {bad}'
    assert 'models/best.pt' in z.namelist()
print('ARTIFACT ZIP:',zip_path,'size MB:',round(zip_path.stat().st_size/1024**2,2),'CRC PASS')
print('Performance:',json.dumps(test_result,indent=2))

## 13 — Before → Now → Next, and claim boundaries

- **Before:** mounted Roboflow YOLO images + source metadata, unverified split safety and possible upstream version overlap.
- **Now (only after successfully executing all cells):** audited annotations, conservative group-disjoint partitions, real YOLOv8n training and validation selection, final held-out test, saved `.pt` model, test predictions, and artifact manifest.
- **Next:** manual error inspection, perceptual duplicate audit, independent-camera data, physical weight/diameter calibration, and backend inference/API with end-to-end tests before a frontend can be called operational.

**Guardrails:** one architecture only; no fabricated benchmark comparisons, no field-grading claims, no ONNX success without smoke test, no production readiness claim. Check complete run logs and all report files before publishing LinkedIn metrics.

### Git placement (after downloading outputs)
- `notebooks/EggSizeGrader_Day38_Kaggle.ipynb`
- `models/` — checkpoint and model card, subject to binary size policy
- `reports/` — JSON/CSV audits and metrics
- `assets/plots/` and `assets/examples/` — generated evidence
- Keep your **fixed repository root structure** unchanged. No `datasets/` folder.